# Scikit-Learn Exercises: Classical ML Muscle Memory — SOLUTION

*Frameworks module — ML & NLP by Data Trainers LLC*

This is the complete solution notebook. Every `None  # YOUR CODE` block is filled in with a working implementation, detailed comments, and notes on common mistakes.

## Learning Objectives (same as exercises)
1. Load and explore built-in sklearn datasets
2. Preprocess features with `StandardScaler` and `PCA`
3. Train and evaluate classification models
4. Perform stratified train-test splits
5. Run k-fold cross-validation
6. Tune hyperparameters with `GridSearchCV`

## Datasets used
- `load_iris` — 150 samples, 4 features, 3 classes
- `load_digits` — 1797 samples, 64 features, 10 classes
- `load_breast_cancer` — 569 samples, 30 features, 2 classes
- `make_classification` — synthetic, fully controllable

## Section 0: Environment Setup

In [ ]:
import sklearn, numpy, matplotlib, seaborn
print(f"scikit-learn : {sklearn.__version__}")
print(f"numpy        : {numpy.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")
print(f"seaborn      : {seaborn.__version__}")
print("Environment ready!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.datasets import (load_iris, load_digits,
                               load_breast_cancer, make_classification)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score)
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)

SEED = 42
np.random.seed(SEED)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
print("All imports successful.")

---
## Section 1: Loading and Exploring Datasets

### Demo (same as exercise)

In [ ]:
# Demo — load iris (identical to exercise)
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
print(f"Shape of X : {X_iris.shape}")
print(f"Shape of y : {y_iris.shape}")
print(f"Class names : {iris.target_names}")
unique, counts = np.unique(y_iris, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  class {label} ({iris.target_names[label]}) : {count} samples")

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(iris.target_names, counts, color=["#5B8DEF", "#FF9F40", "#4ECDC4"])
ax.set_title("Iris — samples per class"); ax.set_ylabel("Count")
plt.tight_layout(); plt.show()

### Solution: Lab 1 — Loading and Exploring Digits

In [ ]:
# Solution: Lab 1 — Loading and Exploring Datasets

# 1. Load the digits dataset
# load_digits() returns a Bunch object with .data, .target, .feature_names, etc.
digits = load_digits()

# 2. Assign feature matrix and labels
# .data is shape (n_samples, n_features); .target is shape (n_samples,)
X_digits = digits.data    # 1797 × 64 (8×8 pixel images flattened)
y_digits = digits.target  # integer labels 0–9

# 3. Print basic statistics
n_samples  = X_digits.shape[0]    # first dimension = number of rows
n_features = X_digits.shape[1]    # second dimension = number of features
# np.unique returns sorted unique values; no return_counts here — just the values
unique_labels = np.unique(y_digits)

print(f"Samples  : {n_samples}")     # Expected: 1797
print(f"Features : {n_features}")    # Expected: 64
print(f"Labels   : {unique_labels}") # Expected: [0 1 2 3 4 5 6 7 8 9]

# 4. Compute counts for the bar chart
# return_counts=True makes np.unique also return how many times each value appears
digit_labels, digit_counts = np.unique(y_digits, return_counts=True)

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(digit_labels, digit_counts, color="#5B8DEF")
ax.set_title("Digits — samples per class")
ax.set_xlabel("Digit"); ax.set_ylabel("Count")
plt.tight_layout(); plt.show()

print(f"Total samples across all classes: {digit_counts.sum()}")  # 1797
# Note: classes are very balanced — each digit has ~180 samples.
# This means accuracy is a fair metric here; no need to worry about imbalance.

# Common mistake: calling load_digits() on every cell instead of storing the result.
# Always store the Bunch in a variable and reuse it.

---
## Section 2: Preprocessing Data

### Demo (same as exercise — StandardScaler + PCA on digits)

In [ ]:
# Demo — StandardScaler on iris (same as exercise)
scaler_demo = StandardScaler()
X_iris_scaled = scaler_demo.fit_transform(X_iris)
print("Before:", round(X_iris[:, 0].mean(), 3), round(X_iris[:, 0].std(), 3))
print("After :", round(X_iris_scaled[:, 0].mean(), 6), round(X_iris_scaled[:, 0].std(), 6))

### Solution: Lab 2 — Preprocessing the Breast Cancer Dataset

In [ ]:
# Solution: Lab 2 — Preprocessing the Breast Cancer Dataset

# 1. Load breast cancer
# This dataset has 569 samples and 30 features (measurements like radius, texture, etc.)
bc = load_breast_cancer()
X_bc = bc.data    # (569, 30)
y_bc = bc.target  # 0=malignant, 1=benign

# 2. Standardize
# CRITICAL: always create a NEW scaler per dataset — never reuse a fitted scaler
# on a different dataset (the means/stds would be wrong).
scaler_bc = StandardScaler()
X_bc_scaled = scaler_bc.fit_transform(X_bc)  # fit AND transform in one step

# 3. Verify scaling
# After standardization each feature should have mean≈0, std≈1
print(f"Before — mean: {X_bc[:, 0].mean():.4f}, std: {X_bc[:, 0].std():.4f}")
print(f"After  — mean: {X_bc_scaled[:, 0].mean():.6f}, std: {X_bc_scaled[:, 0].std():.6f}")
# Expected: mean ≈ 0.000000, std ≈ 1.000000

# 4. PCA to 2 components
# n_components=2 means we keep the 2 directions of maximum variance.
# random_state for reproducibility of the SVD solver.
pca_bc = PCA(n_components=2, random_state=SEED)
X_bc_2d = pca_bc.fit_transform(X_bc_scaled)  # (569, 2)

print(f"\nPCA variance retained: {sum(pca_bc.explained_variance_ratio_)*100:.1f}%")
# Typically ~60-65% for breast cancer with 2 components

# 5. Plot the 2D projection
colors = ["#FF6B6B", "#5B8DEF"]  # malignant=red, benign=blue
fig, ax = plt.subplots(figsize=(7, 5))
for cls_idx, cls_name in enumerate(bc.target_names):
    mask = y_bc == cls_idx
    ax.scatter(X_bc_2d[mask, 0], X_bc_2d[mask, 1],
               c=colors[cls_idx], label=cls_name, alpha=0.6, s=20)
ax.set_title("Breast Cancer — PCA 2D projection")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.legend()
plt.tight_layout(); plt.show()

# Explanation:
# The two classes are quite well-separated in the PCA space — this is a good sign
# that a linear classifier will do well. If the classes overlapped heavily, we'd need
# a non-linear model or more components.
#
# Common mistake: applying PCA on the RAW (unscaled) data. PCA is variance-based —
# features with larger raw scales (like 'mean area' which is in hundreds) would
# completely dominate the principal components. Always scale first.

---
## Section 3: Creating and Evaluating Models

### Solution: Lab 3 — Models and Evaluation on Breast Cancer

In [ ]:
# Solution: Lab 3 — Models and Evaluation on Breast Cancer

# 1. Train-test split — using the scaled data from Lab 2
# stratify=y_bc ensures both splits have the same malignant/benign ratio
X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(
    X_bc_scaled, y_bc,
    test_size=0.2,         # 80% train (455 samples), 20% test (114 samples)
    random_state=SEED,
    stratify=y_bc          # keep class proportions: ~37% malignant, ~63% benign
)
print(f"Train: {len(X_tr_bc)} | Test: {len(X_te_bc)}")

# 2. Create logistic regression
# max_iter=1000: with 30 features the lbfgs solver needs more iterations to converge.
# If you see a ConvergenceWarning, increase max_iter further.
lr_bc = LogisticRegression(max_iter=1000, random_state=SEED)

# 3. Fit on training set
# .fit() learns the decision boundary from X_tr_bc and y_tr_bc only.
# Nothing from X_te_bc is used here — that's the whole point of a split.
lr_bc.fit(X_tr_bc, y_tr_bc)

# 4. Predict on test set
# .predict() returns hard class labels (0 or 1)
y_pred_bc = lr_bc.predict(X_te_bc)

# 5. Evaluate
acc_bc = accuracy_score(y_te_bc, y_pred_bc)
print(f"\nAccuracy: {acc_bc:.4f}  ({acc_bc*100:.1f}%)")
# Expected: ~97%

print("\nClassification report:")
print(classification_report(y_te_bc, y_pred_bc,
                             target_names=bc.target_names, digits=3))
# Key columns: precision (of all predicted positive, how many are right?),
#              recall (of all actual positive, how many did we catch?)
#              f1-score (harmonic mean of precision and recall)

# 6. Confusion matrix
cm_bc = confusion_matrix(y_te_bc, y_pred_bc)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_bc, annot=True, fmt="d", cmap="Blues",
            xticklabels=bc.target_names,
            yticklabels=bc.target_names,
            cbar=False, ax=ax)
ax.set_title(f"Confusion Matrix — LogReg on Breast Cancer (acc={acc_bc:.2f})")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.show()

# Reading this matrix:
# Top-left  = malignant correctly identified as malignant (true positive for malignant)
# Top-right = malignant incorrectly labelled benign (false negative — costly in medicine!)
# Bottom-left = benign incorrectly labelled malignant (false positive — causes anxiety)
# Bottom-right = benign correctly identified as benign
#
# For a medical task, minimising false negatives (top-right cell) is more important
# than minimising false positives. Consider using predict_proba() + a lower threshold.

---
## Section 4: Train-Test Split

### Solution: Lab 4 — Train-Test Split on Digits

In [ ]:
# Solution: Lab 4 — Stratified Train-Test Split on Digits

# 2. Standardize X_digits
# Same process as Lab 2: create scaler, fit_transform on the full dataset.
# Note: in a real project you'd only fit on the train set to avoid leakage.
# For this exercise we fit on all data since we're demonstrating the split mechanics.
scaler_dig = StandardScaler()
X_digits_scaled = scaler_dig.fit_transform(X_digits)  # (1797, 64)

# 3. Stratified 75/25 split
# test_size=0.25 means 25% of 1797 ≈ 449 samples for test; 1348 for train
X_tr_dig, X_te_dig, y_tr_dig, y_te_dig = train_test_split(
    X_digits_scaled, y_digits,
    test_size=0.25,
    random_state=SEED,
    stratify=y_digits  # ensures each digit class is proportionally represented
)

# 4. Print sizes
print(f"Train size : {len(X_tr_dig)}")   # Expected: ~1348
print(f"Test size  : {len(X_te_dig)}")   # Expected: ~449

# 5. Verify stratification
# Each digit should appear in roughly the same % in train and test
print("\nClass distribution (%):")
print(f"{'Digit':<8} {'Train':>8} {'Test':>8}")
for cls in np.unique(y_digits):
    tr_pct = np.sum(y_tr_dig == cls) / len(y_tr_dig) * 100
    te_pct = np.sum(y_te_dig == cls) / len(y_te_dig) * 100
    print(f"  {cls:<6} {tr_pct:>7.1f}% {te_pct:>7.1f}%")
print("\nIf train% ≈ test% for each digit, stratification worked!")

# Explanation:
# Digits is already nearly balanced (~180 samples each), so stratification matters
# less here. But it's a good habit — for imbalanced datasets it's essential.
#
# Common mistake: doing test_size=0.75 instead of 0.25.
# Remember: test_size is the FRACTION that goes to the TEST set.
# So test_size=0.25 means 25% test, 75% train.

---
## Section 5: Cross-Validation

### Solution: Lab 5 — CV on Breast Cancer (LogReg vs RandomForest)

In [ ]:
# Solution: Lab 5 — Cross-Validation on Breast Cancer

# 1. Create a 10-fold stratified CV splitter
# n_splits=10: more folds = lower variance estimate but slower
# shuffle=True: shuffle data before splitting (important when data is sorted by class)
# random_state=SEED: reproducibility of the shuffle
skf_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

# 2. Cross-validate LogisticRegression
# cross_val_score internally clones the estimator for each fold — no data leakage.
# scoring='f1_macro' averages F1 equally across both classes regardless of support.
cv_scores_lr = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=SEED),
    X_bc_scaled,   # full dataset — cross_val_score handles the splitting
    y_bc,
    cv=skf_10,
    scoring='f1_macro'
)

# 3. Cross-validate RandomForestClassifier
# n_estimators=100: 100 trees in the ensemble (more = better but slower)
cv_scores_rf = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=SEED),
    X_bc_scaled,
    y_bc,
    cv=skf_10,
    scoring='f1_macro'
)

# 4. Print results
print("=== 10-Fold Stratified CV (macro F1) ===\n")
print(f"LogisticRegression    : {cv_scores_lr.mean():.4f} ± {cv_scores_lr.std():.4f}")
print(f"RandomForestClassifier: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}")
# Expected: both ~0.96–0.98; they're typically very close on this dataset

# 5. Winner
winner = "LogisticRegression" if cv_scores_lr.mean() > cv_scores_rf.mean() else "RandomForest"
print(f"\nWinner: {winner}")
# On breast cancer, logistic regression often wins or ties with RF because
# the decision boundary is nearly linear after scaling.

# Bonus: box plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([cv_scores_lr, cv_scores_rf],
           labels=["LogReg", "RandomForest"],
           patch_artist=True,
           boxprops=dict(facecolor="#5B8DEF", alpha=0.7))
ax.set_title("CV Macro-F1 distribution (10 folds)")
ax.set_ylabel("Macro F1")
plt.tight_layout(); plt.show()

# Key insight: the std (box width) tells you how stable the model is.
# A model with mean 0.97 ± 0.01 is MORE reliable than one with 0.97 ± 0.05.
# Always report both mean and std in your results.
#
# Common mistake: running cross_val_score on the TEST split (X_te_bc) instead
# of the full dataset. cross_val_score already does its own internal splitting —
# pass the FULL data so it can use all of it.

---
## Section 6: Hyperparameter Tuning

### Solution: Lab 6 — GridSearchCV on Digits with RandomForest

In [ ]:
# Solution: Lab 6 — GridSearchCV with RandomForestClassifier on Digits

# 1. Define parameter grid
# n_estimators: how many trees (more = better accuracy, slower training)
# max_depth: how deep each tree grows (None = unlimited — can overfit)
# 3 × 3 = 9 combinations × 5 folds = 45 model fits total
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20]
}

# 2. Create GridSearchCV
# verbose=1 shows progress; refit=True (default) refits the best model on the full data
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_grid_rf,
    cv=5,                # 5-fold stratified CV (sklearn uses StratifiedKFold for classifiers)
    scoring='f1_macro',
    n_jobs=-1,           # parallelise — all CPU cores
    refit=True           # after search, refit best model on the entire dataset
)

# 3. Fit on full scaled digits dataset
# GridSearchCV handles all internal splitting — we pass ALL the data
gs_rf.fit(X_digits_scaled, y_digits)

# 4. Results
print(f"Best parameters : {gs_rf.best_params_}")
print(f"Best CV F1      : {gs_rf.best_score_:.4f}")
# Expected: typically n_estimators=200, max_depth=None gives best F1 ~0.97+

# Show full grid results table
results_df = pd.DataFrame(gs_rf.cv_results_)
cols = ['param_n_estimators', 'param_max_depth', 'mean_test_score', 'std_test_score', 'rank_test_score']
print("\nAll combinations sorted by rank:")
print(results_df[cols].sort_values('rank_test_score').to_string(index=False))

# 5. Evaluate best estimator on test set (from Lab 4)
# best_estimator_ is already fitted on the full dataset with the best hyperparameters
y_pred_best_rf = gs_rf.best_estimator_.predict(X_te_dig)
acc_best_rf = accuracy_score(y_te_dig, y_pred_best_rf)
print(f"\nTest accuracy (best estimator): {acc_best_rf:.4f}  ({acc_best_rf*100:.1f}%)")
# Expected: ~97-98%

# Explanation:
# GridSearchCV performs nested CV internally. When we call .fit(X_full, y_full),
# it splits X_full into train/val per fold, trains each combo, picks the best,
# then re-trains on ALL of X_full with the best params. That's what best_estimator_ is.
#
# IMPORTANT: even though we called gs_rf.fit(X_digits_scaled, y_digits),
# the best_estimator_ was trained on the FULL X_digits_scaled. So when we evaluate
# on X_te_dig (which is a subset of X_digits_scaled), we're potentially evaluating
# on data that was used during grid search. In production use a separate test set
# held out before GridSearchCV. Here we use it for illustration only.
#
# Alternative: use RandomizedSearchCV when the grid is large — it samples
# combinations randomly instead of exhaustively, saving time.

---
## Optional Lab: Full Pipeline on Synthetic Data — Solution

In [ ]:
# Solution: Optional Lab — Full Pipeline on Synthetic Data

# 1. Generate synthetic dataset
# n_clusters_per_class=1 avoids multimodal classes that are very hard to separate
X_extra, y_extra = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_classes=5, n_clusters_per_class=1, random_state=SEED
)

# 2. Scale
# Same pattern: StandardScaler, fit_transform
scaler_extra = StandardScaler()
X_extra_scaled = scaler_extra.fit_transform(X_extra)  # (2000, 20)

# 3. Stratified 80/20 split
X_tr_ex, X_te_ex, y_tr_ex, y_te_ex = train_test_split(
    X_extra_scaled, y_extra,
    test_size=0.20,
    random_state=SEED,
    stratify=y_extra   # 5 classes — stratification keeps all represented in test
)
print(f"Train: {len(X_tr_ex)} | Test: {len(X_te_ex)}")

# 4. GridSearchCV with GradientBoostingClassifier
# learning_rate controls step size — smaller = more robust but needs more estimators
# n_estimators: boosting rounds
param_grid_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.10]
}
# 2 × 2 × 5 folds = 20 fits — fast enough

gs_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=SEED),
    param_grid_gb,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)
gs_gb.fit(X_tr_ex, y_tr_ex)  # fit on training split only — test set is truly held out

# 5. Results
print(f"\nBest params : {gs_gb.best_params_}")
print(f"Best CV F1  : {gs_gb.best_score_:.4f}")

y_pred_ex = gs_gb.best_estimator_.predict(X_te_ex)
acc_ex = accuracy_score(y_te_ex, y_pred_ex)
print(f"Test accuracy: {acc_ex:.4f}  ({acc_ex*100:.1f}%)")
# Expected: ~75-80% for 5-class synthetic — multiclass with limited features is hard

cm_ex = confusion_matrix(y_te_ex, y_pred_ex)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_ex, annot=True, fmt="d", cmap="Purples", cbar=False, ax=ax)
ax.set_title(f"GradientBoosting — 5-class synthetic (acc={acc_ex:.2f})")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.show()

# This pipeline (scale → split → grid search on train → evaluate on test) is the
# pattern you'll use in every real ML project. The key discipline:
# 1. Never let test data influence any training or tuning step.
# 2. Always scale first, then split (or split first and fit scaler on train only).
# 3. Report CV score (which you optimised) AND test score (which you didn't).

---
## Congratulations — Solutions Complete!

Every exercise in this notebook has a reference implementation above. Here is a concise cheat-sheet of the patterns you practiced:

### sklearn Cheat Sheet

```python
# Load data
from sklearn.datasets import load_iris
iris = load_iris()
X, y = iris.data, iris.target

# Scale
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)         # train data
X_test_scaled = scaler.transform(X_test)   # test data — NEVER fit_transform here

# PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

# Split
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)

# Evaluate
from sklearn.metrics import accuracy_score, classification_report
print(accuracy_score(y_te, y_pred))
print(classification_report(y_te, y_pred))

# Cross-validate
from sklearn.model_selection import cross_val_score, StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=skf, scoring='f1_macro')
print(f"{scores.mean():.4f} ± {scores.std():.4f}")

# Grid search
from sklearn.model_selection import GridSearchCV
gs = GridSearchCV(model, {'C': [0.1, 1, 10]}, cv=5, scoring='f1_macro', n_jobs=-1)
gs.fit(X_tr, y_tr)
print(gs.best_params_, gs.best_score_)
best = gs.best_estimator_
```

### What's next?
These patterns appear in every module. In Module 1 you'll use `LogisticRegression` and `GradientBoostingClassifier` as baselines for text classification — now you know exactly how they work.